# Drug Safety Signal Monitor — Data Ingestion

Run this notebook **directly in Microsoft Fabric** to ingest adverse event data into your Eventhouse.

**No local setup required** — uses Fabric's built-in authentication (zero config).

## Quick Start
1. Update the **Configuration** cell below with your Eventhouse URI
2. Choose a data source: **openFDA API** (live fetch) or **Lakehouse file** (pre-uploaded NDJSON)
3. **Run All Cells** — data loads automatically with progress tracking
4. Check the **Verification** cell output for signal detection results

## Data Sources
| Source | Speed | Internet | Description |
|--------|-------|----------|-------------|
| `openfda` | ~5 min | Required | Fetches real FDA adverse event data live |
| `lakehouse` | ~30 sec | No | Reads pre-generated NDJSON from Lakehouse Files |
| `synthetic` | ~10 sec | No | Generates realistic demo data in-memory |

In [ ]:
# =============================================================
# CONFIGURATION — Update these values for your environment
# =============================================================

CLUSTER_URI = "https://trd-yxvwsau7zstjcxcq7h.z0.kusto.fabric.microsoft.com"
DATABASE    = "Drug-safety-signal-eh"
TABLE       = "AdverseEvents"

# Data source: "openfda", "lakehouse", or "synthetic"
DATA_SOURCE = "openfda"

# openFDA settings (used when DATA_SOURCE = "openfda")
DRUGS = {
    "aspirin":        2000,
    "pembrolizumab":  2000,
    "adalimumab":     2000,
    "warfarin":       1500,
    "metformin":      1000,
    "lisinopril":      800,
    "omeprazole":      700,
}
SPIKE_DRUG = "aspirin"   # Drug with clustered 48h spike for signal detection

# Lakehouse settings (used when DATA_SOURCE = "lakehouse")
LAKEHOUSE_FILE = "Files/adverse_events_portfolio.json"

# Synthetic data settings (used when DATA_SOURCE = "synthetic")
SYNTHETIC_COUNT = 10000

# Ingestion settings
CLEAR_EXISTING = True    # Drop old data before loading
BATCH_SIZE     = 50      # Events per .set-or-append batch

## 1. Install Packages & Connect to Eventhouse

In [ ]:
# Install Kusto SDK (if not already available)
%pip install azure-kusto-data requests -q

import json, uuid, time, datetime, random, sys
from azure.kusto.data import KustoClient, KustoConnectionStringBuilder

# ---- Fabric-native authentication (zero config) ----
try:
    import notebookutils
    def _get_token():
        return notebookutils.credentials.getToken("https://kusto.kusto.windows.net")
    kcsb = KustoConnectionStringBuilder.with_token_provider(CLUSTER_URI, _get_token)
    print("\u2713 Using Fabric built-in authentication")
except ImportError:
    # Fallback for local Jupyter (uses device code)
    from azure.identity import DeviceCodeCredential
    credential = DeviceCodeCredential()
    credential.get_token("https://kusto.kusto.windows.net/.default")
    kcsb = KustoConnectionStringBuilder.with_azure_token_credential(CLUSTER_URI, credential)
    print("\u2713 Authenticated via device code")

client = KustoClient(kcsb)

# Verify connection
result = client.execute(DATABASE, f"{TABLE} | count")
current_count = result.primary_results[0][0][0]
print(f"\u2713 Connected to {DATABASE}.{TABLE}")
print(f"  Current row count: {current_count:,}")

## 2. Load Data

This cell fetches/generates events based on your `DATA_SOURCE` setting above.

In [ ]:
import requests

all_events = []
now = datetime.datetime(2026, 3, 4, 12, 0, 0, tzinfo=datetime.timezone.utc)
base_ts = now - datetime.timedelta(days=30)

# Full 30-column schema (matches Eventhouse table exactly)
TABLE_COLUMNS = [
    ("event_id", "string"), ("safety_report_id", "string"),
    ("receive_date", "datetime"), ("ingest_timestamp", "datetime"),
    ("is_serious", "bool"), ("hospitalization", "bool"),
    ("death", "bool"), ("life_threatening", "bool"),
    ("disability", "bool"), ("congenital_anomaly", "bool"),
    ("required_intervention", "bool"), ("patient_age", "string"),
    ("patient_age_unit", "string"), ("patient_sex", "string"),
    ("patient_weight", "string"), ("drug_name", "string"),
    ("generic_name", "string"), ("brand_name", "string"),
    ("drug_route", "string"), ("drug_indication", "string"),
    ("drug_characterization", "string"), ("reaction_terms", "string"),
    ("primary_reaction", "string"), ("reaction_count", "int"),
    ("country", "string"), ("sender_organization", "string"),
    ("report_type", "string"), ("qualification", "string"),
    ("event_id_1", "guid"), ("drug_characterization_1", "long"),
]


def _parse_fda_date(ds):
    try:
        return datetime.datetime.strptime(ds, "%Y%m%d").strftime("%Y-%m-%dT%H:%M:%SZ")
    except Exception:
        return now.strftime("%Y-%m-%dT%H:%M:%SZ")


def transform_fda_event(raw, drug_override, timestamp_str):
    """Transform a single openFDA event to Eventhouse schema."""
    patient = raw.get("patient", {})
    drugs = patient.get("drug", [])
    suspect = [d for d in drugs if d.get("drugcharacterization") == "1"]
    drug = suspect[0] if suspect else (drugs[0] if drugs else {})

    reactions = patient.get("reaction", [])
    terms = [r.get("reactionmeddrapt", "") for r in reactions if r.get("reactionmeddrapt")]

    openfda = drug.get("openfda", {})
    gn = openfda.get("generic_name", [""])
    bn = openfda.get("brand_name", [""])
    generic = gn[0] if isinstance(gn, list) and gn else str(gn) if gn else ""
    brand   = bn[0] if isinstance(bn, list) and bn else str(bn) if bn else ""

    rd = _parse_fda_date(raw.get("receivedate", ""))

    return {
        "event_id": str(uuid.uuid4()),
        "safety_report_id": raw.get("safetyreportid", str(uuid.uuid4())),
        "receive_date": rd,
        "ingest_timestamp": timestamp_str,
        "is_serious":  raw.get("serious", "2") == "1",
        "hospitalization": raw.get("seriousnesshospitalization", "0") == "1",
        "death": raw.get("seriousnessdeath", "0") == "1",
        "life_threatening": raw.get("seriousnesslifethreatening", "0") == "1",
        "disability": raw.get("seriousnessdisabling", "0") == "1",
        "congenital_anomaly": raw.get("seriousnesscongenitalanomali", "0") == "1",
        "required_intervention": raw.get("seriousnessother", "0") == "1",
        "patient_age": str(patient.get("patientonsetage", "")),
        "patient_age_unit": str(patient.get("patientonsetageunit", "")),
        "patient_sex": str(patient.get("patientsex", "")),
        "patient_weight": str(patient.get("patientweight", "")),
        "drug_name": drug.get("medicinalproduct", ""),
        "generic_name": drug_override,
        "brand_name": brand,
        "drug_route": drug.get("drugadministrationroute", ""),
        "drug_indication": drug.get("drugindication", ""),
        "drug_characterization": drug.get("drugcharacterization", ""),
        "reaction_terms": ",".join(terms),
        "primary_reaction": terms[0] if terms else "",
        "reaction_count": len(terms),
        "country": raw.get("occurcountry", ""),
        "sender_organization": raw.get("sender", {}).get("senderorganization", ""),
        "report_type": raw.get("reporttype", ""),
        "qualification": raw.get("primarysource", {}).get("qualification", ""),
    }


# ---- DATA SOURCE: openFDA API ----
if DATA_SOURCE == "openfda":
    BASE_URL = "https://api.fda.gov/drug/event.json"
    print(f"Fetching ~{sum(DRUGS.values()):,} events from openFDA FAERS API...")
    print(f"Rate limit: 40 req/min (takes ~5 min for 10K events)\n")

    for drug_name, target in DRUGS.items():
        print(f"  {drug_name:20s} target={target:>5,} ", end="", flush=True)
        fetched = []
        for skip in range(0, min(target, 5000), 100):
            try:
                params = {
                    "search": f'patient.drug.openfda.generic_name:\"{drug_name}\" AND receivedate:[20200101 TO 20260101]',
                    "limit": 100, "skip": skip,
                }
                resp = requests.get(BASE_URL, params=params, timeout=30)
                if resp.status_code == 200:
                    fetched.extend(resp.json().get("results", []))
                    if len(fetched) >= target:
                        break
                elif resp.status_code == 429:
                    time.sleep(5)
                    continue
                else:
                    break
                time.sleep(1.6)
            except Exception as e:
                print(f"\n    Warning: {e}")
                time.sleep(3)

        # Transform and assign timestamps
        spike_count = 0
        for i, raw in enumerate(fetched[:target]):
            is_serious = raw.get("serious", "2") == "1"
            # Cluster serious spike-drug events in last 48h
            if drug_name == SPIKE_DRUG and is_serious and spike_count < 30:
                ts = now - datetime.timedelta(hours=random.uniform(1, 48))
                spike_count += 1
            else:
                ts = base_ts + datetime.timedelta(seconds=random.uniform(0, 30 * 86400))
            ts_str = ts.strftime("%Y-%m-%dT%H:%M:%S.%fZ")
            all_events.append(transform_fda_event(raw, drug_name, ts_str))
        print(f"got {len(fetched[:target]):>5,}  (spike_events={spike_count})")

# ---- DATA SOURCE: Lakehouse file ----
elif DATA_SOURCE == "lakehouse":
    print(f"Reading from Lakehouse: {LAKEHOUSE_FILE}")
    import notebookutils
    content = notebookutils.fs.head(LAKEHOUSE_FILE, 1024 * 1024 * 50)  # 50MB max
    for line in content.strip().split("\n"):
        if line.strip():
            all_events.append(json.loads(line))
    print(f"  Loaded {len(all_events):,} events from Lakehouse")

# ---- DATA SOURCE: Synthetic generator ----
elif DATA_SOURCE == "synthetic":
    print(f"Generating {SYNTHETIC_COUNT:,} synthetic adverse events...")
    REACTIONS = [
        "Headache", "Nausea", "Fatigue", "Dizziness", "Rash", "Vomiting",
        "Diarrhoea", "Pyrexia", "Arthralgia", "Cough", "Dyspnoea",
        "Abdominal pain", "Pruritus", "Anaemia", "Thrombocytopenia",
        "Neutropenia", "Pneumonia", "Drug ineffective", "Pain", "Asthenia",
    ]
    COUNTRIES = ["US", "GB", "DE", "FR", "JP", "CA", "AU", "IT", "BR", "IN"]
    ROUTES = ["Oral", "Intravenous", "Subcutaneous", "Topical", "Intramuscular"]
    INDICATIONS = ["Pain", "Cardiology", "Oncology", "Autoimmune", "Diabetes", "Hypertension", "GERD"]
    drug_list = list(DRUGS.keys())

    for i in range(SYNTHETIC_COUNT):
        drug = random.choice(drug_list)
        n_reactions = random.randint(1, 4)
        rxns = random.sample(REACTIONS, n_reactions)
        is_serious = random.random() < 0.75

        # Spike: cluster serious aspirin events in last 48h
        if drug == SPIKE_DRUG and is_serious and i < 30:
            ts = now - datetime.timedelta(hours=random.uniform(1, 48))
        else:
            ts = base_ts + datetime.timedelta(seconds=random.uniform(0, 30 * 86400))

        all_events.append({
            "event_id": str(uuid.uuid4()),
            "safety_report_id": f"SYN-{i:06d}",
            "receive_date": (ts - datetime.timedelta(days=random.randint(1, 60))).strftime("%Y-%m-%dT%H:%M:%SZ"),
            "ingest_timestamp": ts.strftime("%Y-%m-%dT%H:%M:%S.%fZ"),
            "is_serious": is_serious,
            "hospitalization": is_serious and random.random() < 0.4,
            "death": is_serious and random.random() < 0.12,
            "life_threatening": is_serious and random.random() < 0.15,
            "disability": is_serious and random.random() < 0.08,
            "congenital_anomaly": False,
            "required_intervention": is_serious and random.random() < 0.1,
            "patient_age": str(random.choice([25, 35, 45, 55, 65, 72, 80])),
            "patient_age_unit": "801",
            "patient_sex": str(random.choice([1, 2])),
            "patient_weight": str(round(random.gauss(75, 15), 1)),
            "drug_name": drug.upper(),
            "generic_name": drug,
            "brand_name": drug.capitalize(),
            "drug_route": random.choice(ROUTES),
            "drug_indication": random.choice(INDICATIONS),
            "drug_characterization": "1",
            "reaction_terms": ",".join(rxns),
            "primary_reaction": rxns[0],
            "reaction_count": n_reactions,
            "country": random.choice(COUNTRIES),
            "sender_organization": "SYNTHETIC",
            "report_type": str(random.choice([1, 2])),
            "qualification": str(random.choice([1, 2, 3, 5])),
        })

random.shuffle(all_events)
n_serious = sum(1 for e in all_events if e.get("is_serious"))
n_deaths  = sum(1 for e in all_events if e.get("death"))
n_spike   = sum(1 for e in all_events if e.get("generic_name") == SPIKE_DRUG)
print(f"\n\u2713 Prepared {len(all_events):,} events")
print(f"  Serious: {n_serious:,}  |  Deaths: {n_deaths:,}  |  {SPIKE_DRUG}: {n_spike:,}")
print(f"  Unique drugs: {len(set(e['generic_name'] for e in all_events))}")

## 3. Clear Old Data & Ingest

Clears any existing rows (if `CLEAR_EXISTING = True`), then ingests events in batches using `.set-or-append` KQL commands.

In [ ]:
def build_set_or_append(table, events, columns):
    """Build a .set-or-append KQL command for a batch of events."""
    col_defs = ", ".join(f"{n}:{t}" for n, t in columns)
    rows = []
    for evt in events:
        values = []
        for name, ctype in columns:
            val = evt.get(name, "")
            if ctype == "bool":
                values.append("true" if val else "false")
            elif ctype == "int":
                values.append(str(int(val)) if val else "0")
            elif ctype == "long":
                values.append(str(int(val)) if val else "long(null)")
            elif ctype == "datetime":
                values.append(f"datetime({val})" if val else "datetime(null)")
            elif ctype == "guid":
                values.append(f"guid({val})" if val else "guid(null)")
            else:
                safe = str(val).replace("'", "\\'").replace("\n", " ").replace("\r", "")
                values.append(f"'{safe}'")
        rows.append(f"    {', '.join(values)}")
    return f".set-or-append {table} <|\n  datatable({col_defs})\n  [\n" + ",\n".join(rows) + "\n  ]"


# ---- Clear existing data ----
if CLEAR_EXISTING:
    print("Clearing existing data...")
    try:
        client.execute(DATABASE, f".drop extents <| .show table {TABLE} extents")
        print("\u2713 Existing data cleared")
        time.sleep(3)
    except Exception as e:
        print(f"\u26a0 Clear failed (table may be empty): {str(e)[:100]}")

# ---- Batch ingest ----
total = len(all_events)
ingested = 0
errors = 0
start_time = time.time()

print(f"\nIngesting {total:,} events in batches of {BATCH_SIZE}...")

for batch_start in range(0, total, BATCH_SIZE):
    batch = all_events[batch_start : batch_start + BATCH_SIZE]
    try:
        kql = build_set_or_append(TABLE, batch, TABLE_COLUMNS)
        client.execute(DATABASE, kql)
        ingested += len(batch)
    except Exception as e:
        err_msg = str(e)
        if "Forbidden" in err_msg or "Unauthorized" in err_msg:
            print(f"\n\u2717 Authentication error: {err_msg[:200]}")
            print("  Check that your user has Admin or Ingestor role on the database.")
            break
        # Try individual events as fallback
        for evt in batch:
            try:
                kql_single = build_set_or_append(TABLE, [evt], TABLE_COLUMNS)
                client.execute(DATABASE, kql_single)
                ingested += 1
            except Exception:
                errors += 1

    # Progress display
    pct = ingested / total * 100
    elapsed = time.time() - start_time
    rate = ingested / elapsed if elapsed > 0 else 0
    eta = (total - ingested) / rate if rate > 0 else 0
    print(f"\r  [{int(pct):>3d}%] {ingested:>6,}/{total:,}  |  {rate:.0f} evt/s  |  ETA {eta:.0f}s  ", end="", flush=True)

elapsed = time.time() - start_time
print(f"\n\n\u2713 Ingested {ingested:,} of {total:,} events in {elapsed:.1f}s")
if errors:
    print(f"\u26a0 {errors} events failed to ingest")

## 4. Verify & Detect Signals

Run these queries to confirm data loaded correctly and check for safety signals.

In [ ]:
print("=" * 60)
print("  VERIFICATION")
print("=" * 60)

# Total count
r = client.execute(DATABASE, f"{TABLE} | count")
print(f"\n\u2713 Total rows: {r.primary_results[0][0][0]:,}")

# Events by drug
print(f"\n  Events by Drug:")
r = client.execute(DATABASE, f"{TABLE} | summarize count() by generic_name | order by count_ desc")
for row in r.primary_results[0]:
    print(f"    {row['generic_name']:25s} {row['count_']:>6,}")

# Date distribution
print(f"\n  Date Range:")
r = client.execute(DATABASE, f"{TABLE} | summarize min_ts=min(ingest_timestamp), max_ts=max(ingest_timestamp), days=dcount(bin(ingest_timestamp, 1d))")
row = r.primary_results[0][0]
print(f"    From:  {row['min_ts']}")
print(f"    To:    {row['max_ts']}")
print(f"    Days:  {row['days']}")

# Signal Detection
print(f"\n{'=' * 60}")
print(f"  SAFETY SIGNAL DETECTION")
print(f"{'=' * 60}")
try:
    r = client.execute(DATABASE, "DetectSafetySignals(2.0, 3)")
    signals = list(r.primary_results[0])
    print(f"\n  Found {len(signals)} signal(s) above threshold\n")
    # Show top signals sorted by spike_ratio
    top = sorted(signals, key=lambda x: x['spike_ratio'], reverse=True)[:15]
    print(f"  {'Drug':<20s} {'Reaction':<30s} {'Ratio':>8s} {'7d':>5s} {'30d':>5s}")
    print(f"  {'-'*20} {'-'*30} {'-'*8} {'-'*5} {'-'*5}")
    for sig in top:
        drug = str(sig.get('generic_name', ''))[:20]
        rxn  = str(sig.get('primary_reaction', ''))[:30]
        ratio = sig.get('spike_ratio', 0)
        c7   = sig.get('events_7d', 0)
        c30  = sig.get('events_30d', 0)
        flag = ' \u26a0\ufe0f' if drug.lower() == SPIKE_DRUG.lower() else ''
        print(f"  {drug:<20s} {rxn:<30s} {ratio:>8.2f} {c7:>5} {c30:>5}{flag}")
except Exception as e:
    print(f"  DetectSafetySignals function not found: {str(e)[:100]}")
    print(f"  Run the STEP9 setup script to create it.")

print(f"\n\u2713 Data ingestion complete! Open your Eventhouse dashboard to explore.")